# 📊 YouTube Trending Data Analysis
**Aastha Sheth | Data Analysis Portfolio Project**

### What this notebook covers:
1. Load raw JSON data from AWS S3
2. Clean & transform data
3. Trend prediction (views, likes over time)
4. Competitor channel comparison
5. AI-powered insights via Claude API

## 0. Install & Import

In [ ]:
# pip install boto3 pandas numpy matplotlib seaborn scikit-learn anthropic plotly

import boto3, json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import anthropic
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({'figure.facecolor':'#F8FAFC','axes.facecolor':'#F8FAFC',
                     'axes.grid':True,'grid.alpha':0.3,'font.size':11})
print('✅ Imports complete')

## 1. Load Raw JSON from AWS S3

In [ ]:
# ─────────────────────────────────────────────
# CONFIG — update these with your values
# ─────────────────────────────────────────────
S3_BUCKET   = 'your-bucket-name'          # e.g. 'aastha-youtube-data'
S3_PREFIX   = 'raw/trending/'             # folder path inside bucket
AWS_REGION  = 'us-east-1'
ANTHROPIC_API_KEY = 'your-anthropic-key'  # or use os.environ['ANTHROPIC_API_KEY']

# ─────────────────────────────────────────────
# Load from S3
# ─────────────────────────────────────────────
def load_json_from_s3(bucket, prefix, region):
    """Load all JSON files from S3 prefix into a single DataFrame."""
    s3 = boto3.client('s3', region_name=region)
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    all_records = []
    files_loaded = 0
    
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('.json'):
            raw = s3.get_object(Bucket=bucket, Key=obj['Key'])
            data = json.loads(raw['Body'].read().decode('utf-8'))
            
            # Handle both list and dict with 'items' key (YouTube API format)
            if isinstance(data, list):
                all_records.extend(data)
            elif isinstance(data, dict) and 'items' in data:
                all_records.extend(data['items'])
            
            files_loaded += 1
    
    print(f'✅ Loaded {files_loaded} JSON files | {len(all_records)} raw records')
    return all_records

raw_records = load_json_from_s3(S3_BUCKET, S3_PREFIX, AWS_REGION)

## 2. Parse & Clean YouTube API JSON

In [ ]:
def parse_youtube_record(record):
    """
    Parses one YouTube Data API v3 video record.
    Handles the nested snippet / statistics / contentDetails structure.
    """
    snippet    = record.get('snippet', {})
    stats      = record.get('statistics', {})
    content    = record.get('contentDetails', {})
    
    return {
        'video_id'        : record.get('id', ''),
        'title'           : snippet.get('title', ''),
        'channel_title'   : snippet.get('channelTitle', ''),
        'channel_id'      : snippet.get('channelId', ''),
        'published_at'    : snippet.get('publishedAt', ''),
        'category_id'     : snippet.get('categoryId', ''),
        'tags'            : '|'.join(snippet.get('tags', [])),
        'description'     : snippet.get('description', '')[:300],
        'view_count'      : int(stats.get('viewCount', 0)),
        'like_count'      : int(stats.get('likeCount', 0)),
        'comment_count'   : int(stats.get('commentCount', 0)),
        'duration'        : content.get('duration', ''),
        'fetch_date'      : record.get('fetch_date', pd.Timestamp.today().strftime('%Y-%m-%d')),
    }

# Parse all records
df_raw = pd.DataFrame([parse_youtube_record(r) for r in raw_records])

# Convert types
df_raw['published_at'] = pd.to_datetime(df_raw['published_at'], errors='coerce')
df_raw['fetch_date']   = pd.to_datetime(df_raw['fetch_date'],   errors='coerce')
df_raw['days_since_publish'] = (df_raw['fetch_date'] - df_raw['published_at']).dt.days

# Engagement rate
df_raw['engagement_rate'] = ((df_raw['like_count'] + df_raw['comment_count']) 
                              / df_raw['view_count'].replace(0, np.nan) * 100).round(3)

# Category map (YouTube API category IDs)
CAT_MAP = {'1':'Film & Animation','2':'Autos & Vehicles','10':'Music',
           '15':'Pets & Animals','17':'Sports','20':'Gaming',
           '22':'People & Blogs','23':'Comedy','24':'Entertainment',
           '25':'News & Politics','26':'How-to & Style','27':'Education',
           '28':'Science & Tech','29':'Non-profits'}
df_raw['category'] = df_raw['category_id'].astype(str).map(CAT_MAP).fillna('Other')

# Drop duplicates, nulls
df = df_raw.dropna(subset=['view_count','published_at']).drop_duplicates('video_id')
df = df[df['view_count'] > 0].reset_index(drop=True)

print(f'✅ Clean DataFrame: {df.shape[0]} videos | {df.shape[1]} columns')
print(f'   Date range: {df["published_at"].min().date()} → {df["published_at"].max().date()}')
print(f'   Unique channels: {df["channel_title"].nunique()}')
df.head(3)

## 3. Exploratory Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('YouTube Dataset Overview', fontsize=14, fontweight='bold', y=1.02)

# Views distribution
axes[0].hist(np.log1p(df['view_count']), bins=40, color='#02809090', edgecolor='white')
axes[0].set_title('Views Distribution (log scale)')
axes[0].set_xlabel('log(View Count)')

# Top categories
cat_counts = df['category'].value_counts().head(8)
axes[1].barh(cat_counts.index, cat_counts.values, color='#1E276180')
axes[1].set_title('Videos by Category')
axes[1].invert_yaxis()

# Engagement rate boxplot
top_cats = df['category'].value_counts().head(5).index
df[df['category'].isin(top_cats)].boxplot(
    column='engagement_rate', by='category', ax=axes[2],
    patch_artist=True, flierprops={'marker':'.'}
)
axes[2].set_title('Engagement Rate by Category')
axes[2].set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key stats:')
print(df[['view_count','like_count','comment_count','engagement_rate']].describe().round(1))

## 4. Trend Prediction — Views & Likes Over Time

In [ ]:
# ── Aggregate by week ──────────────────────────────────────────
df['week'] = df['published_at'].dt.to_period('W').dt.start_time
weekly = (df.groupby('week')
            .agg(avg_views=('view_count','median'),
                 avg_likes=('like_count','median'),
                 video_count=('video_id','count'))
            .reset_index()
            .sort_values('week'))

weekly = weekly[weekly['video_count'] >= 3]  # drop weeks with < 3 videos

# ── Polynomial regression trend line ──────────────────────────
def fit_trend(x_dates, y_values, degree=3):
    """Fit polynomial trend and return predictions + R2."""
    x_num = (x_dates - x_dates.min()).dt.days.values.reshape(-1,1)
    model = Pipeline([('poly', PolynomialFeatures(degree)),
                      ('reg',  LinearRegression())])
    model.fit(x_num, y_values)
    preds = model.predict(x_num)
    r2 = r2_score(y_values, preds)
    return preds, r2, model, x_num

view_preds, view_r2, view_model, x_num = fit_trend(weekly['week'], weekly['avg_views'])
like_preds, like_r2, _, _             = fit_trend(weekly['week'], weekly['avg_likes'])

# ── Future forecast (8 weeks) ─────────────────────────────────
last_x    = x_num[-1][0]
future_x  = np.array([[last_x + 7*i] for i in range(1,9)])
future_dates = pd.date_range(weekly['week'].max() + pd.Timedelta('7D'), periods=8, freq='W')
future_views = view_model.predict(future_x)

# ── Plot ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
fig.suptitle('Trend Prediction: Views & Likes Over Time', fontsize=14, fontweight='bold')

# Views
ax1.plot(weekly['week'], weekly['avg_views'], 'o-', color='#1E2761', alpha=0.6,
         linewidth=1.5, markersize=4, label='Actual median views')
ax1.plot(weekly['week'], view_preds, '--', color='#02C39A', linewidth=2.5,
         label=f'Trend line (R²={view_r2:.2f})')
ax1.plot(future_dates, future_views, 's--', color='#F59E0B', linewidth=2,
         markersize=5, label='8-week forecast')
ax1.fill_between(future_dates, future_views*0.85, future_views*1.15,
                  alpha=0.15, color='#F59E0B', label='±15% confidence band')
ax1.set_ylabel('Median Views'); ax1.legend(loc='upper left')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))

# Likes
ax2.plot(weekly['week'], weekly['avg_likes'], 'o-', color='#6D28D9', alpha=0.6,
         linewidth=1.5, markersize=4, label='Actual median likes')
ax2.plot(weekly['week'], like_preds, '--', color='#02C39A', linewidth=2.5,
         label=f'Trend line (R²={like_r2:.2f})')
ax2.set_ylabel('Median Likes'); ax2.legend(loc='upper left')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e3:.0f}K'))

for ax in [ax1,ax2]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('trend_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Views trend R²: {view_r2:.3f}')
print(f'Likes trend R²: {like_r2:.3f}')
print(f'8-week view forecast range: {future_views.min():,.0f} – {future_views.max():,.0f}')

## 5. Competitor Channel Comparison

In [ ]:
# ── Pick top N channels by total views ──────────────────────────
TOP_N = 8

channel_stats = (df.groupby('channel_title')
                   .agg(
                       total_views    = ('view_count',    'sum'),
                       median_views   = ('view_count',    'median'),
                       total_likes    = ('like_count',    'sum'),
                       avg_engagement = ('engagement_rate','mean'),
                       video_count    = ('video_id',      'count'),
                       avg_comments   = ('comment_count', 'mean'),
                       categories     = ('category',      lambda x: x.mode()[0]),
                   )
                   .reset_index()
                   .sort_values('total_views', ascending=False)
                   .head(TOP_N)
                   .reset_index(drop=True))

channel_stats['views_per_video'] = (channel_stats['total_views'] 
                                     / channel_stats['video_count']).round(0)
channel_stats['likes_per_video'] = (channel_stats['total_likes'] 
                                     / channel_stats['video_count']).round(0)

print('Top channels by total views:')
display(channel_stats[['channel_title','total_views','median_views',
                         'avg_engagement','video_count','views_per_video']].round(2))

In [ ]:
# ── Competitor Comparison Dashboard ──────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Total Views by Channel',
        'Avg Engagement Rate (%)',
        'Views per Video (Efficiency)',
        'Video Count vs Median Views'
    ]
)

colors = px.colors.qualitative.Set2
channels = channel_stats['channel_title'].tolist()

# Total views bar
fig.add_trace(go.Bar(
    x=channels, y=channel_stats['total_views'],
    marker_color=colors[:TOP_N], name='Total Views',
    text=(channel_stats['total_views']/1e6).round(1).astype(str)+'M',
    textposition='outside'
), row=1, col=1)

# Engagement rate bar
fig.add_trace(go.Bar(
    x=channels, y=channel_stats['avg_engagement'].round(2),
    marker_color=colors[:TOP_N], name='Engagement %',
    text=channel_stats['avg_engagement'].round(2).astype(str)+'%',
    textposition='outside'
), row=1, col=2)

# Views per video
fig.add_trace(go.Bar(
    x=channels, y=channel_stats['views_per_video'],
    marker_color=colors[:TOP_N], name='Views/Video',
), row=2, col=1)

# Scatter: video count vs median views (bubble = engagement)
fig.add_trace(go.Scatter(
    x=channel_stats['video_count'],
    y=channel_stats['median_views'],
    mode='markers+text',
    text=channel_stats['channel_title'],
    textposition='top center',
    marker=dict(
        size=channel_stats['avg_engagement']*10,
        color=colors[:TOP_N],
        line=dict(width=1, color='white')
    ),
    name='Channels'
), row=2, col=2)

fig.update_layout(
    height=700, showlegend=False,
    title_text='Competitor Channel Analysis Dashboard',
    title_font_size=16,
    paper_bgcolor='#F8FAFC',
    plot_bgcolor='#F8FAFC',
    font=dict(size=10)
)
fig.update_xaxes(tickangle=-35)
fig.write_html('competitor_dashboard.html')
fig.show()
print('✅ Saved competitor_dashboard.html')

In [ ]:
# ── Monthly trend per channel ─────────────────────────────────
df['month'] = df['published_at'].dt.to_period('M').dt.start_time
top_channel_names = channel_stats['channel_title'].tolist()

channel_monthly = (df[df['channel_title'].isin(top_channel_names)]
                     .groupby(['month','channel_title'])
                     .agg(avg_views=('view_count','median'))
                     .reset_index())

fig2 = px.line(
    channel_monthly, x='month', y='avg_views',
    color='channel_title',
    title='Monthly View Trends — Top Channels',
    labels={'avg_views':'Median Views','month':'Month','channel_title':'Channel'},
    template='plotly_white'
)
fig2.update_layout(height=450, paper_bgcolor='#F8FAFC')
fig2.write_html('channel_trends.html')
fig2.show()
print('✅ Saved channel_trends.html')

## 6. AI Insights via Claude API

In [ ]:
def build_analysis_summary(df, channel_stats, weekly, view_r2, future_views):
    """Build a structured data summary to send to Claude."""
    top5 = channel_stats.head(5)
    
    summary = f"""
YOUTUBE TRENDING DATA ANALYSIS SUMMARY
=======================================
Dataset: {len(df)} videos | {df['channel_title'].nunique()} unique channels
Date range: {df['published_at'].min().date()} to {df['published_at'].max().date()}

OVERALL METRICS:
- Median views per video: {df['view_count'].median():,.0f}
- Median likes per video: {df['like_count'].median():,.0f}
- Avg engagement rate: {df['engagement_rate'].mean():.2f}%
- Most common category: {df['category'].mode()[0]}

TREND ANALYSIS:
- Views trend R² score: {view_r2:.3f} ({'strong' if view_r2>0.7 else 'moderate' if view_r2>0.4 else 'weak'} fit)
- 8-week view forecast: {future_views.min():,.0f} to {future_views.max():,.0f} median views
- Trend direction: {'upward' if future_views[-1] > future_views[0] else 'downward'}

TOP 5 CHANNELS BY TOTAL VIEWS:
""" + top5[['channel_title','total_views','avg_engagement','views_per_video','video_count']].to_string(index=False) + """

CATEGORY BREAKDOWN (top 5 by video count):
""" + df['category'].value_counts().head(5).to_string() + """

ENGAGEMENT LEADERS (top 3 by engagement rate, min 5 videos):
""" + (df.groupby('channel_title')
          .filter(lambda x: len(x)>=5)
          .groupby('channel_title')['engagement_rate']
          .mean()
          .sort_values(ascending=False)
          .head(3)
          .to_string())
    
    return summary

summary_text = build_analysis_summary(df, channel_stats, weekly, view_r2, future_views)
print(summary_text)

In [ ]:
def get_claude_insights(summary_text, api_key, analysis_type='full'):
    """
    Send analysis summary to Claude API and get structured insights.
    analysis_type: 'full' | 'trends' | 'competitors' | 'recommendations'
    """
    client = anthropic.Anthropic(api_key=api_key)
    
    prompts = {
        'full': f"""
You are a senior data analyst specializing in digital media and YouTube content strategy.
Analyze the following YouTube trending data summary and provide:

1. KEY FINDINGS (3 bullet points — most important patterns)
2. COMPETITOR INSIGHTS (which channel strategy is winning and why)
3. TREND INTERPRETATION (what the view/like trends mean for content creators)
4. ACTIONABLE RECOMMENDATIONS (3 specific, data-backed actions)
5. RISK FLAGS (anything concerning in the data)

Be specific, cite numbers from the data, and keep each section to 2-3 sentences.

DATA SUMMARY:
{summary_text}
""",
        'trends': f"""
You are a data analyst. Based on this YouTube data summary, focus ONLY on:
- What the view/like trends reveal about audience behavior
- Whether the 8-week forecast is reliable (comment on R² score)
- What content creators should do based on these trends

DATA: {summary_text}
""",
        'competitors': f"""
You are a competitive intelligence analyst. Based on this YouTube data:
- Which channel has the best strategy (views vs engagement tradeoff)?
- What separates the top performer from others?
- What 2 tactics should a new channel copy from the leader?

DATA: {summary_text}
"""
    }
    
    message = client.messages.create(
        model       = 'claude-sonnet-4-20250514',
        max_tokens  = 1000,
        messages    = [{'role': 'user', 'content': prompts[analysis_type]}]
    )
    
    return message.content[0].text

# ── Get full analysis
print('🤖 Sending data to Claude API...\n')
full_insights     = get_claude_insights(summary_text, ANTHROPIC_API_KEY, 'full')
trend_insights    = get_claude_insights(summary_text, ANTHROPIC_API_KEY, 'trends')
competitor_insights = get_claude_insights(summary_text, ANTHROPIC_API_KEY, 'competitors')

print('=' * 60)
print('CLAUDE FULL ANALYSIS')
print('=' * 60)
print(full_insights)

print('\n' + '=' * 60)
print('TREND INSIGHTS')
print('=' * 60)
print(trend_insights)

print('\n' + '=' * 60)
print('COMPETITOR INSIGHTS')
print('=' * 60)
print(competitor_insights)

In [ ]:
# ── Save all insights to JSON for the dashboard ──────────────
insights_data = {
    'generated_at'       : pd.Timestamp.now().isoformat(),
    'full_analysis'      : full_insights,
    'trend_insights'     : trend_insights,
    'competitor_insights': competitor_insights,
    'metrics': {
        'total_videos'   : len(df),
        'total_channels' : df['channel_title'].nunique(),
        'median_views'   : int(df['view_count'].median()),
        'avg_engagement' : round(df['engagement_rate'].mean(), 3),
        'view_trend_r2'  : round(view_r2, 3),
        'top_channel'    : channel_stats.iloc[0]['channel_title'],
        'top_category'   : df['category'].mode()[0],
    }
}

with open('ai_insights.json', 'w') as f:
    json.dump(insights_data, f, indent=2)

print('✅ Saved ai_insights.json — ready for Streamlit dashboard')
print('\nProject files generated:')
for fname in ['overview.png','trend_prediction.png',
               'competitor_dashboard.html','channel_trends.html','ai_insights.json']:
    print(f'  ✓ {fname}')